# OpenFoodFacts Global Nutrition Analysis

## Notebook 2: Data Cleaning and Preprocessing

### Objective

The objective of this notebook is to clean and prepare the Open Food Facts dataset for exploratory data analysis. This includes examining missing values in greater detail, removing unnecessary columns, handling incomplete records, correcting data types where required, and creating a cleaner dataset for subsequent analysis.

### Workflow

- Import the required libraries
- Load the dataset
- Create a working copy of the dataset
- Examine missing values in detail
- Remove columns with excessive missing values
- Select the variables required for analysis
- Handle missing values in important columns
- Check and correct data types
- Remove invalid records
- Save the cleaned dataset
- Summarize the preprocessing steps

## Importing the Required Libraries

In this section, we import the Python libraries required for data cleaning and preprocessing. These libraries provide tools for data manipulation, numerical operations, and basic visualization that will be used throughout this notebook.

In [1]:
# Import the required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Configuring Notebook Display Settings

To improve readability during data exploration, we configure a few display options that allow more columns and longer text values to be displayed without truncation.

In [2]:
# Configure notebook display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 100)

## Loading the Dataset

The Open Food Facts dataset is loaded into a Pandas DataFrame. This provides the starting point for data cleaning and preprocessing before performing exploratory analysis.

In [3]:
# load the dataset

food_df = pd.read_csv("../data/raw/en.openfoodfacts.org.products.tsv", sep = "\t", low_memory = False)

## Creating a Working Copy of the Dataset

Instead of modifying the original dataset directly, we create a separate working copy. This preserves the original data and allows us to perform cleaning operations safely without affecting the source dataset.

In [4]:
# Create a working copy of the dataset

clean_df = food_df.copy()

## Examining Missing Values in Detail

Before cleaning the dataset, it is important to understand the extent of missing data across all variables. This helps identify columns that may require removal, imputation, or further investigation before analysis.

In [5]:
# Calculate the number of missing values in each column

missing_values = clean_df.isnull().sum()
missing_values

code                                              26
url                                               26
creator                                            3
created_t                                          3
created_datetime                                  10
last_modified_t                                    0
last_modified_datetime                             0
product_name                                   17512
generic_name                                  298313
quantity                                      236742
packaging                                     266068
packaging_tags                                266068
brands                                         29050
brands_tags                                    29070
categories                                    252728
categories_tags                               252752
categories_en                                 252726
origins                                       330977
origins_tags                                  

## Identifying Columns with Missing Values

To focus on incomplete variables, we display only the columns that contain one or more missing values. This provides a clearer view of the features that require preprocessing.

In [6]:
# Display only the columns with missing values

missing_values[missing_values > 0]

code                                              26
url                                               26
creator                                            3
created_t                                          3
created_datetime                                  10
product_name                                   17512
generic_name                                  298313
quantity                                      236742
packaging                                     266068
packaging_tags                                266068
brands                                         29050
brands_tags                                    29070
categories                                    252728
categories_tags                               252752
categories_en                                 252726
origins                                       330977
origins_tags                                  331015
manufacturing_places                          314019
manufacturing_places_tags                     

## Ranking Missing Values by Column

Sorting the missing values in descending order makes it easier to identify the variables with the highest proportion of missing data. This information will guide decisions on whether to retain or remove specific columns.

In [7]:
# Sort missing values in descending order

missing_values.sort_values(ascending = False)

ingredients_from_palm_oil                     356027
ingredients_that_may_be_from_palm_oil         356027
no_nutriments                                 356027
chlorophyl_100g                               356027
water-hardness_100g                           356027
glycemic-index_100g                           356027
-butyric-acid_100g                            356027
-melissic-acid_100g                           356027
-nervonic-acid_100g                           356027
-erucic-acid_100g                             356027
-mead-acid_100g                               356027
-elaidic-acid_100g                            356027
-caproic-acid_100g                            356027
-lignoceric-acid_100g                         356027
-cerotic-acid_100g                            356027
nutrition_grade_uk                            356027
-montanic-acid_100g                           356026
-palmitic-acid_100g                           356026
-stearic-acid_100g                            

## Calculating the Percentage of Missing Values

The number of missing values alone does not provide a complete picture of data quality. Calculating the percentage of missing values for each column makes it easier to compare variables and identify those that contain a high proportion of incomplete data.

In [8]:
# Calculate the percentage of missing values

missing_percentage = (missing_values / len(clean_df)) * 100
missing_percentage

code                                            0.007303
url                                             0.007303
creator                                         0.000843
created_t                                       0.000843
created_datetime                                0.002809
last_modified_t                                 0.000000
last_modified_datetime                          0.000000
product_name                                    4.918728
generic_name                                   83.789432
quantity                                       66.495519
packaging                                      74.732534
packaging_tags                                 74.732534
brands                                          8.159494
brands_tags                                     8.165111
categories                                     70.985627
categories_tags                                70.992369
categories_en                                  70.985066
origins                        

## Identifying Columns with High Missing Percentages

Columns with a very high percentage of missing values often contribute little to the analysis and may negatively affect data quality. Reviewing these percentages helps determine which variables should be considered for removal during preprocessing.

In [9]:
# Sort missing value percentages in descending order

missing_percentage.sort_values(ascending = False)

ingredients_from_palm_oil                     100.000000
ingredients_that_may_be_from_palm_oil         100.000000
no_nutriments                                 100.000000
chlorophyl_100g                               100.000000
water-hardness_100g                           100.000000
glycemic-index_100g                           100.000000
-butyric-acid_100g                            100.000000
-melissic-acid_100g                           100.000000
-nervonic-acid_100g                           100.000000
-erucic-acid_100g                             100.000000
-mead-acid_100g                               100.000000
-elaidic-acid_100g                            100.000000
-caproic-acid_100g                            100.000000
-lignoceric-acid_100g                         100.000000
-cerotic-acid_100g                            100.000000
nutrition_grade_uk                            100.000000
-montanic-acid_100g                            99.999719
-palmitic-acid_100g            

## Defining a Missing Value Threshold

Not every column with missing values needs to be removed. To make the preprocessing process consistent and objective, a missing value threshold is defined. Columns exceeding this threshold will be considered for removal, while important variables with moderate missing values will be reviewed individually before making a final decision.

In [10]:
# Identify columns with more than 90% missing values

columns_to_remove = missing_percentage[missing_percentage > 90].index
columns_to_remove

Index(['origins', 'origins_tags', 'emb_codes', 'emb_codes_tags',
       'first_packaging_code_geo', 'cities', 'cities_tags', 'allergens_en',
       'traces', 'traces_tags', 'traces_en', 'no_nutriments',
       'ingredients_from_palm_oil', 'ingredients_from_palm_oil_tags',
       'ingredients_that_may_be_from_palm_oil',
       'ingredients_that_may_be_from_palm_oil_tags', 'nutrition_grade_uk',
       'energy-from-fat_100g', '-butyric-acid_100g', '-caproic-acid_100g',
       '-caprylic-acid_100g', '-capric-acid_100g', '-lauric-acid_100g',
       '-myristic-acid_100g', '-palmitic-acid_100g', '-stearic-acid_100g',
       '-arachidic-acid_100g', '-behenic-acid_100g', '-lignoceric-acid_100g',
       '-cerotic-acid_100g', '-montanic-acid_100g', '-melissic-acid_100g',
       'monounsaturated-fat_100g', 'polyunsaturated-fat_100g',
       'omega-3-fat_100g', '-alpha-linolenic-acid_100g',
       '-eicosapentaenoic-acid_100g', '-docosahexaenoic-acid_100g',
       'omega-6-fat_100g', '-linoleic-aci

In [11]:
# Keep potassium_100g even though it exceeds the missing value threshold

columns_to_remove = columns_to_remove.drop("potassium_100g")

In [12]:
# Display the final list of columns to be removed

columns_to_remove

Index(['origins', 'origins_tags', 'emb_codes', 'emb_codes_tags',
       'first_packaging_code_geo', 'cities', 'cities_tags', 'allergens_en',
       'traces', 'traces_tags', 'traces_en', 'no_nutriments',
       'ingredients_from_palm_oil', 'ingredients_from_palm_oil_tags',
       'ingredients_that_may_be_from_palm_oil',
       'ingredients_that_may_be_from_palm_oil_tags', 'nutrition_grade_uk',
       'energy-from-fat_100g', '-butyric-acid_100g', '-caproic-acid_100g',
       '-caprylic-acid_100g', '-capric-acid_100g', '-lauric-acid_100g',
       '-myristic-acid_100g', '-palmitic-acid_100g', '-stearic-acid_100g',
       '-arachidic-acid_100g', '-behenic-acid_100g', '-lignoceric-acid_100g',
       '-cerotic-acid_100g', '-montanic-acid_100g', '-melissic-acid_100g',
       'monounsaturated-fat_100g', 'polyunsaturated-fat_100g',
       'omega-3-fat_100g', '-alpha-linolenic-acid_100g',
       '-eicosapentaenoic-acid_100g', '-docosahexaenoic-acid_100g',
       'omega-6-fat_100g', '-linoleic-aci

## Removing Columns with Excessive Missing Values

After reviewing the missing value percentages, columns containing more than 90% missing data are removed from the working dataset. These variables provide very limited information for the current analysis and are unlikely to contribute meaningful insights.

In [13]:
# Remove columns with excessive missing values

clean_df = clean_df.drop(columns = columns_to_remove)

In [14]:
# Display the updated dataset dimensions

clean_df.shape

(356027, 64)

## Reviewing Missing Values After Column Removal

After removing columns with excessive missing values, we examine the remaining dataset to understand which variables still contain missing data. This helps identify the columns that require further preprocessing before analysis.

In [15]:
# Calculate missing values in the cleaned dataset

remaining_missing_values = clean_df.isnull().sum()
remaining_missing_values[remaining_missing_values > 0].sort_values(ascending = False)

potassium_100g                             331179
allergens                                  318851
manufacturing_places_tags                  314026
manufacturing_places                       314019
stores                                     298326
generic_name                               298313
labels                                     296929
labels_tags                                296849
labels_en                                  296823
purchase_places                            289462
image_url                                  280302
image_small_url                            280302
packaging_tags                             266068
packaging                                  266068
main_category                              252778
main_category_en                           252778
categories_tags                            252752
categories                                 252728
categories_en                              252726
quantity                                   236742


## Selecting the Core Variables for Analysis

The cleaned dataset still contains several descriptive and nutritional variables. For the purposes of this project, we identify the core variables that are most relevant to food product information and nutritional analysis. These variables will form the basis of the exploratory analysis in the following notebook.

In [16]:
# Display the remaining column names

list(clean_df.columns)

['code',
 'url',
 'creator',
 'created_t',
 'created_datetime',
 'last_modified_t',
 'last_modified_datetime',
 'product_name',
 'generic_name',
 'quantity',
 'packaging',
 'packaging_tags',
 'brands',
 'brands_tags',
 'categories',
 'categories_tags',
 'categories_en',
 'manufacturing_places',
 'manufacturing_places_tags',
 'labels',
 'labels_tags',
 'labels_en',
 'purchase_places',
 'stores',
 'countries',
 'countries_tags',
 'countries_en',
 'ingredients_text',
 'allergens',
 'serving_size',
 'additives_n',
 'additives',
 'additives_tags',
 'additives_en',
 'ingredients_from_palm_oil_n',
 'ingredients_that_may_be_from_palm_oil_n',
 'nutrition_grade_fr',
 'pnns_groups_1',
 'pnns_groups_2',
 'states',
 'states_tags',
 'states_en',
 'main_category',
 'main_category_en',
 'image_url',
 'image_small_url',
 'energy_100g',
 'fat_100g',
 'saturated-fat_100g',
 'trans-fat_100g',
 'cholesterol_100g',
 'carbohydrates_100g',
 'sugars_100g',
 'fiber_100g',
 'proteins_100g',
 'salt_100g',
 'sodiu

## Verifying the Data Types

After removing columns with excessive missing values, we verify the data types of the remaining variables. Ensuring that each column has an appropriate data type helps prevent errors during analysis and supports accurate data processing.

In [17]:
# Display the data types of the cleaned dataset

clean_df.dtypes

code                                        object
url                                         object
creator                                     object
created_t                                   object
created_datetime                            object
last_modified_t                             object
last_modified_datetime                      object
product_name                                object
generic_name                                object
quantity                                    object
packaging                                   object
packaging_tags                              object
brands                                      object
brands_tags                                 object
categories                                  object
categories_tags                             object
categories_en                               object
manufacturing_places                        object
manufacturing_places_tags                   object
labels                         

## Checking for Duplicate Records After Preprocessing

After removing columns with excessive missing values, we verify whether duplicate records are present in the cleaned dataset. Ensuring that duplicate entries are absent helps maintain data quality before exploratory analysis.

In [18]:
# Count duplicate records in the cleaned dataset

clean_duplicates = clean_df.duplicated().sum()
print(f"Number of duplicate records: {clean_duplicates}")

Number of duplicate records: 0


## Saving the Cleaned Dataset

After completing the preprocessing steps, the cleaned dataset is saved to the processed data folder. This dataset will be used as the input for the exploratory data analysis performed in the next notebook.

In [19]:
# Save the cleaned dataset

clean_df.to_csv("../data/processed/openfoodfacts_cleaned.csv", index = False)

## Preprocessing Summary

The preprocessing stage focused on improving the quality and usability of the dataset while preserving the original information required for analysis. Columns with excessive missing values were removed based on a predefined threshold, the remaining variables were reviewed, data types were verified, and duplicate records were checked. The resulting cleaned dataset provides a reliable foundation for the exploratory data analysis presented in the next notebook.

In [20]:
# Display a summary of the preprocessing results

print(f"Original dataset shape : {food_df.shape}")
print(f"Cleaned dataset shape  : {clean_df.shape}")
print(f"Columns removed        : {food_df.shape[1] - clean_df.shape[1]}")
print(f"Duplicate records      : {clean_duplicates}")

Original dataset shape : (356027, 163)
Cleaned dataset shape  : (356027, 64)
Columns removed        : 99
Duplicate records      : 0
